In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from mlflow.models.signature import infer_signature
import pandas as pd
import numpy as np
from utils import eval_metrics


In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target
X.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
alpha = 0.5
l1_ratio = 0.5

with mlflow.start_run(run_name='manual_run'):
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    rmse, mae, r2 = eval_metrics(y_test, preds)

    mlflow.log_param('alpha', alpha)
    mlflow.log_param('l1_ratio', l1_ratio)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('r2', r2)

    mlflow.sklearn.log_model(model, 'model')

(rmse, mae, r2)


In [ ]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name='autolog_run'):
    model = ElasticNet(alpha=0.3, l1_ratio=0.7)
    model.fit(X_train, y_train)


In [ ]:
alphas = [0.1, 0.5, 1.0]
l1_ratios = [0.2, 0.5, 0.8]

for a in alphas:
    for l in l1_ratios:
        with mlflow.start_run(run_name=f'a={a}_l={l}'):
            model = ElasticNet(alpha=a, l1_ratio=l)
            model.fit(X_train, y_train)

            preds = model.predict(X_test)
            rmse, mae, r2 = eval_metrics(y_test, preds)

            mlflow.log_param('alpha', a)
            mlflow.log_param('l1_ratio', l)
            mlflow.log_metric('rmse', rmse)
            mlflow.log_metric('mae', mae)
            mlflow.log_metric('r2', r2)

            mlflow.sklearn.log_model(model, 'model')
